# 02 — Experimental evaluation: top-down vs top-down + bottom-up

Questo notebook **non genera goal** e **non esegue il feedback loop**. Legge due insiemi di output già persistiti:

- `output/top_down/`: baseline top-down;
- `output/top_down_bottom_up/`: lo stesso stato iniziale dopo il ciclo bottom-up.

Per ciascun dataset confronta separatamente HLG e LLG contro la ground truth usando la stessa metrica semantica BERT/Hungarian già presente nel notebook di valutazione precedente. Calcola quindi **Precision, Recall e F1** per:

1. top-down only;
2. top-down + bottom-up;
3. delta bottom-up − top-down.

Il risultato permette di verificare direttamente se il feedback bottom-up migliora o peggiora la baseline, senza mescolare generazione ed evaluation nello stesso notebook.

In [19]:
from pathlib import Path
import json
import os
import socket
import sys

os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")

import nltk
import numpy as np
import pandas as pd
import torch
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_BOTTOM_UP_DIR = OUTPUT_ROOT / "top_down_bottom_up"
EVALUATION_DIR = OUTPUT_ROOT / "evaluation" / "top_down_vs_bottom_up"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

from groundtruth import (
    GENOME,
    GESTAO_HOSPITAL,
    SIA_PROJECT_25_26,
    LONDON_AMBULANCE_SYSTEM,
)

GROUNDTRUTHS = [
    SIA_PROJECT_25_26,
    GENOME,
    GESTAO_HOSPITAL,
    LONDON_AMBULANCE_SYSTEM,
]
GROUNDTRUTH_BY_NAME = {item["name"]: item for item in GROUNDTRUTHS}

print("Project root:", PROJECT_ROOT)
print("Top-down directory:", TOP_DOWN_DIR)
print("Top-down + bottom-up directory:", TOP_DOWN_BOTTOM_UP_DIR)
print("Evaluation output:", EVALUATION_DIR)

Project root: C:\Users\agnes_lryeu3v\Desktop\tesi
Top-down directory: C:\Users\agnes_lryeu3v\Desktop\tesi\output\top_down
Top-down + bottom-up directory: C:\Users\agnes_lryeu3v\Desktop\tesi\output\top_down_bottom_up
Evaluation output: C:\Users\agnes_lryeu3v\Desktop\tesi\output\evaluation\top_down_vs_bottom_up


## 1. Caricamento e pairing degli output

Il pairing viene fatto sul campo `name` del JSON, non sulla posizione dei file. Un dataset entra nel confronto solo se sono disponibili sia la baseline sia l'output con feedback e se esiste la relativa ground truth.

In [20]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def index_json_files(directory: Path, pattern: str):
    result = {}
    for path in sorted(directory.glob(pattern)):
        payload = load_json(path)
        dataset_name = payload.get("name")
        if not dataset_name:
            print(f"Skipping file without dataset name: {path}")
            continue
        if dataset_name in result:
            raise RuntimeError(
                f"More than one file found for dataset {dataset_name!r} in {directory}. "
                "Remove stale duplicates or narrow the file pattern."
            )
        result[dataset_name] = (path, payload)
    return result


baseline_by_dataset = index_json_files(TOP_DOWN_DIR, "*_top_down.json")
combined_by_dataset = index_json_files(TOP_DOWN_BOTTOM_UP_DIR, "*_top_down_bottom_up.json")

common_datasets = sorted(
    set(baseline_by_dataset)
    & set(combined_by_dataset)
    & set(GROUNDTRUTH_BY_NAME)
)

if not common_datasets:
    raise RuntimeError(
        "No complete baseline/feedback pair found. Run the two execution notebooks first."
    )

print(f"Comparable datasets: {len(common_datasets)}")
for dataset_name in common_datasets:
    print(
        f"- {dataset_name}\n"
        f"    baseline: {baseline_by_dataset[dataset_name][0].name}\n"
        f"    feedback: {combined_by_dataset[dataset_name][0].name}"
    )

missing_feedback = sorted(set(baseline_by_dataset) - set(combined_by_dataset))
if missing_feedback:
    print("\nBaselines without a bottom-up result:", missing_feedback)

Comparable datasets: 4
- Genome Nexus
    baseline: Genome Nexus_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: Genome Nexus_ZERO_SHOT_feedback_original_top_down_top_down_bottom_up.json
- Gestao Hospital
    baseline: Gestao Hospital_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: Gestao Hospital_ZERO_SHOT_feedback_original_top_down_top_down_bottom_up.json
- London Ambulance Service
    baseline: London Ambulance Service_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: London Ambulance Service_ZERO_SHOT_feedback_original_top_down_top_down_bottom_up.json
- SIA Project 25 26
    baseline: SIA Project 25 26_ZERO_SHOT_feedback_original_top_down_top_down.json
    feedback: SIA Project 25 26_ZERO_SHOT_feedback_original_top_down_top_down_bottom_up.json


## 2. Estrazione delle collezioni di goal

La baseline usa le liste `highLevelGoals` e `lowLevelGoals`. L'output con feedback usa invece le collezioni finali restituite dall'orchestrator (`finalHighLevelGoals`, `finalLowLevelGoals`).

In [21]:
def goal_text(item) -> str:
    if item is None:
        return ""
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        description = item.get("description")
        if isinstance(description, str) and description.strip():
            return description.strip()
        name = item.get("name")
        if isinstance(name, str):
            return name.strip()
    return str(item).strip()


def extract_hlg(container):
    if container is None:
        return []
    items = container.get("goals", []) if isinstance(container, dict) else container
    return [text for item in items if (text := goal_text(item))]


def extract_llg(container):
    if container is None:
        return []
    items = container.get("low_level_goals", []) if isinstance(container, dict) else container
    return [text for item in items if (text := goal_text(item))]


def extract_cycle_status(payload):
    cycle = payload.get("globalGoalCycle") or {}
    return {
        "converged": bool(cycle.get("converged")),
        "stop_reason": cycle.get("stop_reason"),
        "completed_iterations": cycle.get("completed_iterations"),
    }


states = {}
for dataset_name in common_datasets:
    _, baseline = baseline_by_dataset[dataset_name]
    _, combined = combined_by_dataset[dataset_name]
    gt = GROUNDTRUTH_BY_NAME[dataset_name]

    states[dataset_name] = {
        "reference_hlg": [str(x).strip() for x in gt["highLevelGoals"] if str(x).strip()],
        "reference_llg": [str(x).strip() for x in gt["lowLevelGoals"] if str(x).strip()],
        "top_down_hlg": extract_hlg(baseline.get("highLevelGoals")),
        "top_down_llg": extract_llg(baseline.get("lowLevelGoals")),
        "feedback_hlg": extract_hlg(combined.get("finalHighLevelGoals")),
        "feedback_llg": extract_llg(combined.get("finalLowLevelGoals")),
        "cycle_status": extract_cycle_status(combined),
    }

for dataset_name, s in states.items():
    print(
        f"{dataset_name}: "
        f"GT HLG/LLG={len(s['reference_hlg'])}/{len(s['reference_llg'])} | "
        f"top-down={len(s['top_down_hlg'])}/{len(s['top_down_llg'])} | "
        f"+bottom-up={len(s['feedback_hlg'])}/{len(s['feedback_llg'])} | "
        f"converged={s['cycle_status']['converged']}"
    )

Genome Nexus: GT HLG/LLG=9/34 | top-down=3/8 | +bottom-up=3/11 | converged=True
Gestao Hospital: GT HLG/LLG=4/19 | top-down=5/14 | +bottom-up=5/18 | converged=True
London Ambulance Service: GT HLG/LLG=2/10 | top-down=5/12 | +bottom-up=5/12 | converged=False
SIA Project 25 26: GT HLG/LLG=9/21 | top-down=5/18 | +bottom-up=5/26 | converged=False


## 3. Preparazione della metrica BERT/Hungarian

La metrica è la stessa logica usata nel precedente notebook sperimentale:

- preprocessing NLTK;
- embedding `bert-base-uncased` con mean pooling;
- cosine similarity tra goal generati e goal di riferimento;
- matching 1-a-1 ottimo tramite algoritmo ungherese;
- soft Precision = similarità totale / numero di goal generati;
- soft Recall = similarità totale / numero di goal di riferimento;
- F1 armonica di Precision e Recall.

In [22]:
NLTK_RESOURCES = {
    "punkt": ("tokenizers/punkt",),
    "punkt_tab": ("tokenizers/punkt_tab",),
    "wordnet": ("corpora/wordnet", "corpora/wordnet.zip"),
    "omw-1.4": ("corpora/omw-1.4", "corpora/omw-1.4.zip"),
    "stopwords": ("corpora/stopwords",),
}


def _resource_exists(candidates):
    for candidate in candidates:
        try:
            nltk.data.find(candidate)
            return True
        except LookupError:
            pass
    return False


previous_timeout = socket.getdefaulttimeout()
socket.setdefaulttimeout(45)
try:
    for package, candidates in NLTK_RESOURCES.items():
        if not _resource_exists(candidates):
            print(f"Downloading missing NLTK resource: {package}")
            ok = nltk.download(package, quiet=False, raise_on_error=False)
            if not ok:
                raise RuntimeError(f"Could not download NLTK resource: {package}")
finally:
    socket.setdefaulttimeout(previous_timeout)

BERT_MODEL_NAME = "bert-base-uncased"
BERT_BATCH_SIZE = 16

print(f"Loading {BERT_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModel.from_pretrained(BERT_MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

print("Device:", device)

Loading bert-base-uncased...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cpu


In [23]:
class PaperAlignedGoalEvaluator:
    def __init__(self, tokenizer, model, device, batch_size=16, preprocess=True):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.batch_size = batch_size
        self.preprocess = preprocess
        self.embedding_cache = {}

    def preprocess_text(self, text):
        tokens = word_tokenize(text)
        filtered_tokens = [
            token for token in tokens
            if token not in stop_words and token.isalnum()
        ]
        stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]
        lemmatized_tokens = [lemmatizer.lemmatize(token) for token in stemmed_tokens]
        return " ".join(lemmatized_tokens)

    def _encode_missing(self, raw_texts):
        missing = [text for text in raw_texts if text not in self.embedding_cache]
        if not missing:
            return

        prepared = [
            self.preprocess_text(text) if self.preprocess else text
            for text in missing
        ]

        total_batches = (len(missing) + self.batch_size - 1) // self.batch_size
        for start in tqdm(
            range(0, len(missing), self.batch_size),
            total=total_batches,
            desc="BERT batches",
        ):
            raw_batch = missing[start:start + self.batch_size]
            text_batch = prepared[start:start + self.batch_size]

            encoded_input = self.tokenizer(
                text_batch,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)

            with torch.inference_mode():
                model_output = self.model(**encoded_input)

            token_embeddings = model_output.last_hidden_state
            attention_mask = encoded_input["attention_mask"]
            input_mask_expanded = (
                attention_mask.unsqueeze(-1)
                .expand(token_embeddings.size())
                .float()
            )

            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
            sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
            pooled = (sum_embeddings / sum_mask).detach().cpu()

            for text, embedding in zip(raw_batch, pooled):
                self.embedding_cache[text] = embedding

    def encode_texts(self, texts):
        self._encode_missing(texts)
        return torch.stack([self.embedding_cache[text] for text in texts])

    def compute_similarity(self, generated_goals, reference_goals):
        gen_embeddings = self.encode_texts(generated_goals)
        ref_embeddings = self.encode_texts(reference_goals)
        gen_norm = torch.nn.functional.normalize(gen_embeddings, p=2, dim=1)
        ref_norm = torch.nn.functional.normalize(ref_embeddings, p=2, dim=1)
        return torch.mm(gen_norm, ref_norm.transpose(0, 1)).numpy()

    def calculate_soft_f1(self, preds, refs):
        if not preds or not refs:
            return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

        sim_matrix = self.compute_similarity(preds, refs)
        row_ind, col_ind = linear_sum_assignment(-sim_matrix)
        total_similarity = float(sim_matrix[row_ind, col_ind].sum())

        precision = total_similarity / len(preds)
        recall = total_similarity / len(refs)
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        return {"precision": precision, "recall": recall, "f1": f1}


evaluator = PaperAlignedGoalEvaluator(
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=BERT_BATCH_SIZE,
    preprocess=True,
)

## 4. Precision, Recall e F1 separati per baseline e feedback

Ogni riga usa la stessa ground truth e cambia solo lo stato generato. Questo è il confronto principale da usare per rispondere alla domanda sperimentale: **il bottom-up migliora la qualità dei goal rispetto alla stessa pipeline top-down senza feedback?**

In [24]:
metric_rows = []

for dataset_name, s in states.items():
    status = s["cycle_status"]
    for level, reference, top_down, feedback in (
        (
            "High-Level Goals",
            s["reference_hlg"],
            s["top_down_hlg"],
            s["feedback_hlg"],
        ),
        (
            "Low-Level Goals",
            s["reference_llg"],
            s["top_down_llg"],
            s["feedback_llg"],
        ),
    ):
        for method, generated in (
            ("top_down", top_down),
            ("top_down_plus_bottom_up", feedback),
        ):
            m = evaluator.calculate_soft_f1(generated, reference)
            metric_rows.append(
                {
                    "dataset": dataset_name,
                    "level": level,
                    "method": method,
                    "generated_goal_count": len(generated),
                    "reference_goal_count": len(reference),
                    "precision": m["precision"],
                    "recall": m["recall"],
                    "f1": m["f1"],
                    "cycle_converged": status["converged"],
                    "cycle_stop_reason": status["stop_reason"],
                    "cycle_completed_iterations": status["completed_iterations"],
                }
            )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(4))

BERT batches:   0%|          | 0/1 [00:00<?, ?it/s]

BERT batches:   0%|          | 0/1 [00:00<?, ?it/s]

BERT batches:   0%|          | 0/1 [00:00<?, ?it/s]

BERT batches:   0%|          | 0/1 [00:00<?, ?it/s]

BERT batches:   0%|          | 0/3 [00:00<?, ?it/s]

...

BERT batches:   0%|          | 0/1 [00:00<?, ?it/s]

BERT batches:   0%|          | 0/2 [00:00<?, ?it/s]

BERT batches:   0%|          | 0/2 [00:00<?, ?it/s]

BERT batches:   0%|          | 0/2 [00:00<?, ?it/s]

,dataset,level,method,generated_goal_count,reference_goal_count,precision,recall,f1,cycle_converged,cycle_stop_reason,cycle_completed_iterations
0,Genome Nexus,High-Level Goals,top_down,3,9,0.8273,0.2758,0.4136,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
1,Genome Nexus,High-Level Goals,top_down_plus_bottom_up,3,9,0.8187,0.2729,0.4094,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
2,Genome Nexus,Low-Level Goals,top_down,8,34,0.8419,0.1981,0.3207,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
3,Genome Nexus,Low-Level Goals,top_down_plus_bottom_up,11,34,0.8481,0.2744,0.4146,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
4,Gestao Hospital,High-Level Goals,top_down,5,4,0.6387,0.7984,0.7097,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
5,Gestao Hospital,High-Level Goals,top_down_plus_bottom_up,5,4,0.6397,0.7996,0.7108,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
6,Gestao Hospital,Low-Level Goals,top_down,14,19,0.8024,0.5912,0.6808,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
7,Gestao Hospital,Low-Level Goals,top_down_plus_bottom_up,18,19,0.7681,0.7276,0.7473,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED,2
8,London Ambulance Service,High-Level Goals,top_down,5,2,0.3096,0.7741,0.4423,False,HIGH_LEVEL_REGENERATION_FAILED,2
9,London Ambulance Service,High-Level Goals,top_down_plus_bottom_up,5,2,0.2976,0.7440,0.4251,False,HIGH_LEVEL_REGENERATION_FAILED,2


## 5. Delta bottom-up − top-down

Un delta positivo indica miglioramento; un delta negativo indica peggioramento. Precision e Recall vanno interpretati insieme: ad esempio, aggiungere molti goal può aumentare Recall ma ridurre Precision.

In [25]:
baseline_df = (
    metrics_df[metrics_df["method"] == "top_down"]
    .set_index(["dataset", "level"])
)
feedback_df = (
    metrics_df[metrics_df["method"] == "top_down_plus_bottom_up"]
    .set_index(["dataset", "level"])
)

comparison_rows = []
for key in baseline_df.index.intersection(feedback_df.index):
    dataset_name, level = key
    base = baseline_df.loc[key]
    feedback = feedback_df.loc[key]
    comparison_rows.append(
        {
            "dataset": dataset_name,
            "level": level,
            "top_down_goal_count": int(base["generated_goal_count"]),
            "bottom_up_goal_count": int(feedback["generated_goal_count"]),
            "delta_goal_count": int(feedback["generated_goal_count"] - base["generated_goal_count"]),
            "top_down_precision": base["precision"],
            "bottom_up_precision": feedback["precision"],
            "delta_precision": feedback["precision"] - base["precision"],
            "top_down_recall": base["recall"],
            "bottom_up_recall": feedback["recall"],
            "delta_recall": feedback["recall"] - base["recall"],
            "top_down_f1": base["f1"],
            "bottom_up_f1": feedback["f1"],
            "delta_f1": feedback["f1"] - base["f1"],
            "cycle_converged": bool(feedback["cycle_converged"]),
            "cycle_stop_reason": feedback["cycle_stop_reason"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

print("\nRows where bottom-up improved F1:")
display(comparison_df[comparison_df["delta_f1"] > 0].sort_values("delta_f1", ascending=False).round(4))

print("\nRows where bottom-up worsened F1:")
display(comparison_df[comparison_df["delta_f1"] < 0].sort_values("delta_f1").round(4))

,dataset,level,top_down_goal_count,bottom_up_goal_count,delta_goal_count,top_down_precision,bottom_up_precision,delta_precision,top_down_recall,bottom_up_recall,delta_recall,top_down_f1,bottom_up_f1,delta_f1,cycle_converged,cycle_stop_reason
0,Genome Nexus,High-Level Goals,3,3,0,0.8273,0.8187,-0.0085,0.2758,0.2729,-0.0028,0.4136,0.4094,-0.0043,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED
1,Genome Nexus,Low-Level Goals,8,11,3,0.8419,0.8481,0.0062,0.1981,0.2744,0.0763,0.3207,0.4146,0.0939,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED
2,Gestao Hospital,High-Level Goals,5,5,0,0.6387,0.6397,0.0010,0.7984,0.7996,0.0012,0.7097,0.7108,0.0011,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED
3,Gestao Hospital,Low-Level Goals,14,18,4,0.8024,0.7681,-0.0343,0.5912,0.7276,0.1364,0.6808,0.7473,0.0665,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED
4,London Ambulance Service,High-Level Goals,5,5,0,0.3096,0.2976,-0.0121,0.7741,0.7440,-0.0301,0.4423,0.4251,-0.0172,False,HIGH_LEVEL_REGENERATION_FAILED
5,London Ambulance Service,Low-Level Goals,12,12,0,0.6178,0.6026,-0.0153,0.7414,0.7231,-0.0184,0.6740,0.6573,-0.0167,False,HIGH_LEVEL_REGENERATION_FAILED
6,SIA Project 25 26,High-Level Goals,5,5,0,0.6915,0.6608,-0.0306,0.3841,0.3671,-0.0170,0.4939,0.4720,-0.0219,False,MAX_ITERATIONS_REACHED
7,SIA Project 25 26,Low-Level Goals,18,26,8,0.6676,0.5397,-0.1279,0.5722,0.6683,0.0960,0.6163,0.5972,-0.0191,False,MAX_ITERATIONS_REACHED



Rows where bottom-up improved F1:


,dataset,level,top_down_goal_count,bottom_up_goal_count,delta_goal_count,top_down_precision,bottom_up_precision,delta_precision,top_down_recall,bottom_up_recall,delta_recall,top_down_f1,bottom_up_f1,delta_f1,cycle_converged,cycle_stop_reason
1,Genome Nexus,Low-Level Goals,8,11,3,0.8419,0.8481,0.0062,0.1981,0.2744,0.0763,0.3207,0.4146,0.0939,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED
3,Gestao Hospital,Low-Level Goals,14,18,4,0.8024,0.7681,-0.0343,0.5912,0.7276,0.1364,0.6808,0.7473,0.0665,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED
2,Gestao Hospital,High-Level Goals,5,5,0,0.6387,0.6397,0.0010,0.7984,0.7996,0.0012,0.7097,0.7108,0.0011,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED



Rows where bottom-up worsened F1:


,dataset,level,top_down_goal_count,bottom_up_goal_count,delta_goal_count,top_down_precision,bottom_up_precision,delta_precision,top_down_recall,bottom_up_recall,delta_recall,top_down_f1,bottom_up_f1,delta_f1,cycle_converged,cycle_stop_reason
6,SIA Project 25 26,High-Level Goals,5,5,0,0.6915,0.6608,-0.0306,0.3841,0.3671,-0.0170,0.4939,0.4720,-0.0219,False,MAX_ITERATIONS_REACHED
7,SIA Project 25 26,Low-Level Goals,18,26,8,0.6676,0.5397,-0.1279,0.5722,0.6683,0.0960,0.6163,0.5972,-0.0191,False,MAX_ITERATIONS_REACHED
4,London Ambulance Service,High-Level Goals,5,5,0,0.3096,0.2976,-0.0121,0.7741,0.7440,-0.0301,0.4423,0.4251,-0.0172,False,HIGH_LEVEL_REGENERATION_FAILED
5,London Ambulance Service,Low-Level Goals,12,12,0,0.6178,0.6026,-0.0153,0.7414,0.7231,-0.0184,0.6740,0.6573,-0.0167,False,HIGH_LEVEL_REGENERATION_FAILED
0,Genome Nexus,High-Level Goals,3,3,0,0.8273,0.8187,-0.0085,0.2758,0.2729,-0.0028,0.4136,0.4094,-0.0043,True,ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED


## 6. Riassunto macro tra dataset

La macro-media attribuisce lo stesso peso a ogni dataset. È utile come sintesi, mentre le righe per dataset restano necessarie per vedere dove il ciclo aiuta o danneggia la decomposizione.

In [26]:
macro_df = (
    metrics_df
    .groupby(["level", "method"], as_index=False)[["precision", "recall", "f1"]]
    .mean()
)

display(macro_df.round(4))

macro_pivot = macro_df.pivot(index="level", columns="method", values=["precision", "recall", "f1"])
print("\nMacro comparison:")
display(macro_pivot.round(4))

,level,method,precision,recall,f1
0,High-Level Goals,top_down,0.6168,0.5581,0.5149
1,High-Level Goals,top_down_plus_bottom_up,0.6042,0.5459,0.5043
2,Low-Level Goals,top_down,0.7324,0.5257,0.5729
3,Low-Level Goals,top_down_plus_bottom_up,0.6896,0.5983,0.6041



Macro comparison:


precision                           recall  \
method            top_down top_down_plus_bottom_up top_down   
level                                                         
High-Level Goals    0.6168                  0.6042   0.5581   
Low-Level Goals     0.7324                  0.6896   0.5257   

                                               f1                          
method           top_down_plus_bottom_up top_down top_down_plus_bottom_up  
level                                                                      
High-Level Goals                  0.5459   0.5149                  0.5043  
Low-Level Goals                   0.5983   0.5729                  0.6041

## 7. Export

I CSV seguenti sono già separati tra metriche assolute e delta A/B, così possono essere usati direttamente nelle tabelle della tesi.

In [27]:
metrics_path = EVALUATION_DIR / "top_down_vs_bottom_up_metrics.csv"
comparison_path = EVALUATION_DIR / "top_down_vs_bottom_up_deltas.csv"
macro_path = EVALUATION_DIR / "top_down_vs_bottom_up_macro_summary.csv"
summary_path = EVALUATION_DIR / "evaluation_summary.json"

metrics_df.to_csv(metrics_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
macro_df.to_csv(macro_path, index=False)

summary = {
    "datasets": common_datasets,
    "metric": {
        "embedding_model": BERT_MODEL_NAME,
        "matching": "Hungarian maximum-weight one-to-one matching on cosine similarity",
        "preprocessing": "NLTK tokenization + stopword filtering + Porter stemming + WordNet lemmatization",
    },
    "outputs": {
        "metrics_csv": str(metrics_path),
        "deltas_csv": str(comparison_path),
        "macro_summary_csv": str(macro_path),
    },
}
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

for path in (metrics_path, comparison_path, macro_path, summary_path):
    print("Saved:", path)

Saved: C:\Users\agnes_lryeu3v\Desktop\tesi\output\evaluation\top_down_vs_bottom_up\top_down_vs_bottom_up_metrics.csv
Saved: C:\Users\agnes_lryeu3v\Desktop\tesi\output\evaluation\top_down_vs_bottom_up\top_down_vs_bottom_up_deltas.csv
Saved: C:\Users\agnes_lryeu3v\Desktop\tesi\output\evaluation\top_down_vs_bottom_up\top_down_vs_bottom_up_macro_summary.csv
Saved: C:\Users\agnes_lryeu3v\Desktop\tesi\output\evaluation\top_down_vs_bottom_up\evaluation_summary.json
